# Exploratory Data Analysis (EDA) - Islamabad AQI
This notebook explores historical weather and air pollution metrics in Islamabad to identify patterns and trends that will guide model training.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to path to fetch from local Feature Store
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
from pipelines.feature_store import get_feature_store

sns.set_theme(style="darkgrid")

## 1. Load Feature Store Dataset

In [ ]:
fs = get_feature_store()
aqi_fg = fs.get_feature_group("aqi_predictions_fg", version=1)
df = aqi_fg.read()

# Filter for Islamabad
df_isb = df[df['city'].str.lower() == 'islamabad'] if 'city' in df.columns else df
df_isb['timestamp'] = pd.to_datetime(df_isb['timestamp'])
df_isb = df_isb.sort_values(by='timestamp').reset_index(drop=True)

print(f"Loaded {len(df_isb)} records of Islamabad air quality features.")
df_isb.head()

## 2. Statistical Analysis & Feature Distributions

In [ ]:
df_isb[['aqi', 'pm25', 'pm10', 'no2', 'temp', 'humidity']].describe()

## 3. Correlation Matrix
Let's see how weather parameters and individual pollutants correlate with the total AQI.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df_isb[['aqi', 'pm25', 'pm10', 'no2', 'temp', 'humidity', 'wind_speed', 'pressure']].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Islamabad Weather & Pollutant Correlations")
plt.show()

## 4. AQI Distribution Over Hours of Day
Checking diurnal variations to see if traffic hours or weather dynamics increase pollution.

In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_isb, x='hour', y='aqi', marker='o', errorbar=None, color='tomato')
plt.title("Average Hourly AQI Pattern in Islamabad")
plt.xlabel("Hour of Day")
plt.ylabel("Mean AQI")
plt.show()